# Chapter 17 — Debugging What You Cannot See

**Book alignment:** Debugging AI From First Principles, Chapter 17

**Question this notebook isolates:** The refund answer is wrong, repeatably, at temperature
0 — and there is no interior to inspect. Does a boundary-first probe (freeze the bundle,
move **one** crossing, ≥5 trials, prediction pre-written) separate **H1** (retrieval missing
section 4.2) from **H2** (system prompt prefers the general policy) from **H3** (weights)?
And is the model's own explanation admissible as a trace?

In [ ]:
# an OPAQUE model: deterministic given the boundary, but nothing inside is exposable.
def opaque_model(*, context_has_4_2, system_prefers_general, weights_can_apply, case_seed):
    if not context_has_4_2:
        return case_seed % 6 == 0            # ~2/12 correct by luck
    if system_prefers_general:
        return case_seed % 5 == 0            # instruction suppresses the exception
    if not weights_can_apply:
        return False
    return case_seed % 12 != 0               # ~11/12 correct once the boundary is right

FIXTURE = list(range(12))                    # 12 split-shipment cases

def score(**boundary):
    return sum(opaque_model(case_seed=c, **boundary) for c in FIXTURE)

## 1. Freeze the bundle; state the behaviour as a count

In [ ]:
BUNDLE = dict(context_has_4_2=False, system_prefers_general=False, weights_can_apply=True)
baseline = [score(**BUNDLE) for _ in range(5)]         # 5 trials, temperature 0 -> identical
print("baseline pass count (5 trials):", baseline)
assert len(set(baseline)) == 1 and baseline[0] <= 3
print("criterion: cites policy 4.2-exception AND computes the right amount; currently 2/12")

## 2. One boundary crossing moves — repaired retrieval — with a pre-written FORECAST

In [ ]:
# FORECAST  H1: repaired-context >= 10/12    H2: stays <= 4/12    H3: stays <= 4/12
probe = [score(**{**BUNDLE, "context_has_4_2": True}) for _ in range(5)]
print("repaired-context pass count (5 trials):", probe)
assert min(probe) >= 10
print("matches the H1 forecast -> retrieval owns this fixture; H2/H3 suspended, not deleted")

# counter-check: had the system prompt been the cause, the same probe would NOT have moved
h2_world = [score(context_has_4_2=True, system_prefers_general=True, weights_can_apply=True) for _ in range(5)]
assert max(h2_world) <= 4
print("(in an H2 world the identical probe stays ~2/12 - that is what makes it discriminating)")

## 3. The model's explanation is another output, not a trace

In [ ]:
def answer_with_explanation(**boundary):
    ok = opaque_model(case_seed=0, **boundary)
    return {"correct": ok,
            "explanation": "I consulted the refund policy and applied the 30-day rule."}

base = answer_with_explanation(**BUNDLE)
corrupted = dict(base, explanation="I ignored all documents and guessed randomly.")
assert corrupted["correct"] == base["correct"]        # behaviour independent of the stated reason
print("explanation corrupted, behaviour unchanged -> narration, not mechanism (Ch3's load-bearing test)")

## What we earned

Uninspectable is not undiagnosable. With the interior dark, the diagnosis lives at the
boundary: freeze the bundle (revision, input bytes, params, seed), move exactly one crossing,
run a trial series, and check the count against a prediction written first. The repaired-
context probe moved 2/12 → 11/12 exactly as the H1 forecast said; an H2 cause would have
left the same probe flat. The model's fluent explanation changed nothing when corrupted —
so it is filed as output, never as a trace.

**Notebook 18 / Chapter 18** narrows "the boundary" from a continent to a layer: is the
model actually the problem, or is everything around it?